# 1049. Last Stone Weight II - Learning Guide

## Problem Statement
You are given an array of integers `stones` where `stones[i]` is the weight of the ith stone.

We are playing a game with the stones. On each turn, we choose any two stones and smash them together. Suppose the stones have weights x and y with x <= y. The result of this smash is:
- If x == y, both stones are destroyed
- If x != y, the stone of weight x is destroyed, and the stone of weight y has new weight y - x

At the end of the game, there is at most one stone left.

**Return the smallest possible weight of the left stone. If there are no stones left, return 0.**

### Examples
```
Input: stones = [2,7,4,1,8,1]
Output: 1

Input: stones = [31,26,33,21,40]
Output: 5
```

## 🔑 Key Insight: Transform to Subset Sum Problem

### The Trick
When we smash stones, we're essentially **assigning + or - signs** to each stone!

Example: `[2,7,4]`
- Assignment 1: `+2 +7 -4 = 5`
- Assignment 2: `+2 -7 +4 = -1` → absolute value = 1
- Assignment 3: `-2 +7 -4 = 1`

### Mathematical Formulation
Split stones into two groups S1 and S2:
- Result = |sum(S1) - sum(S2)|
- Total = sum(S1) + sum(S2)

**Goal:** Find subset S1 with sum closest to Total/2
- If sum(S1) ≈ Total/2, then sum(S2) ≈ Total/2
- Result = |sum(S1) - sum(S2)| ≈ 0

**Formula:** Result = Total - 2 × maxSum (where maxSum ≤ Total/2)

---
# Method 1: DP Array Approach

## Concept
`dp[i]` = True if we can achieve sum `i` using some subset of stones

### Why Traverse Backwards?
To avoid using the same stone twice in one iteration!

```
Forward (WRONG): stone=3, dp[3]=True, then dp[6]=True (used stone twice!)
Backward (CORRECT): Updates don't affect current iteration
```

## Exercise 1.1: Initialize DP Array
Create the DP array and set base case.

In [1]:
def initialize_dp(target):
    # TODO: Create dp array of size (target + 1) filled with False
    # Set dp[0] = True (we can always make sum 0 with empty subset)
    # Return the dp array
    dp = [False]*(target+1)
    dp[0] = True

    return dp

# Test
dp = initialize_dp(10)
print(f"DP array: {dp}")
print(f"Expected: [True, False, False, False, False, False, False, False, False, False, False]")

DP array: [True, False, False, False, False, False, False, False, False, False, False]
Expected: [True, False, False, False, False, False, False, False, False, False, False]


## Exercise 1.2: Update DP for Single Stone
Given a DP array and one stone, update all achievable sums.

In [4]:
def update_dp_single_stone(dp, stone):
    # TODO: Update dp array for one stone
    # Traverse from (len(dp)-1) down to stone
    # For each index i: dp[i] = dp[i] or dp[i - stone]
    target = len(dp) - 1
    
    for i in range(target, stone - 1, -1):
        dp[i] = dp[i] or dp[i-stone]
    
    return dp


# Test
dp = [True] + [False] * 10  # target = 10
print(f"Before: {dp}")
update_dp_single_stone(dp, 3)
print(f"After adding stone=3: {dp}")
print(f"Expected: dp[3] should be True")

# Add another stone
update_dp_single_stone(dp, 5)
print(f"After adding stone=5: {dp}")
print(f"Expected: dp[3], dp[5], dp[8] should be True")

Before: [True, False, False, False, False, False, False, False, False, False, False]
After adding stone=3: [True, False, False, True, False, False, False, False, False, False, False]
Expected: dp[3] should be True
After adding stone=5: [True, False, False, True, False, True, False, False, True, False, False]
Expected: dp[3], dp[5], dp[8] should be True


## Exercise 1.3: Find Maximum Achievable Sum
Given a completed DP array, find the largest sum that's achievable.

In [8]:
def find_max_sum(dp):
    # TODO: Traverse dp array from end to start
    # Return the first index i where dp[i] is True
    target = len(dp) - 1

    for i in range(target-1, -1,-1):
        if dp[i]:
            return i
        

# Test
dp = [True, False, True, True, False, True, False, False, True, False, False]
print(f"DP array: {dp}")
print(f"Max achievable sum: {find_max_sum(dp)}")
print(f"Expected: 8 (largest index with True)")

DP array: [True, False, True, True, False, True, False, False, True, False, False]
Max achievable sum: 8
Expected: 8 (largest index with True)


## Exercise 1.4: Complete DP Solution
Now put it all together!

In [10]:
def lastStoneWeightII_dp_array(stones):
    # TODO: Implement the complete solution
    # 1. Calculate total and target
    # 2. Initialize dp array
    # 3. For each stone, update dp array (backwards)
    # 4. Find max achievable sum
    # 5. Return total - 2 * max_sum
    total = sum(stones)
    target = total // 2
    dp = [True] + [False] * target

    for stone in stones:
        for i in range(target, stone-1,-1):
            dp[i] = dp[i] or dp[i-stone]
    
    for i in range(target, -1,-1):
        if dp[i]:
            return total - 2 * i

# Test
print(lastStoneWeightII_dp_array([2,7,4,1,8,1]))  # Expected: 1
print(lastStoneWeightII_dp_array([31,26,33,21,40]))  # Expected: 5

1
5


## 📝 Solution 1: DP Array (Reference)

In [11]:
def lastStoneWeightII_array(stones):
    total = sum(stones)
    target = total // 2
    
    dp = [False] * (target + 1)
    dp[0] = True
    
    for stone in stones:
        for i in range(target, stone - 1, -1):
            dp[i] = dp[i] or dp[i - stone]
    
    for i in range(target, -1, -1):
        if dp[i]:
            return total - 2 * i
    
    return total

# Test
print("DP Array Solution:")
print(lastStoneWeightII_array([2,7,4,1,8,1]))  # 1
print(lastStoneWeightII_array([31,26,33,21,40]))  # 5

DP Array Solution:
1
5


---
# Method 2: Set-Based DP Approach

## Concept
Instead of boolean array, use a **set** to store all achievable sums.

### Advantages
- More intuitive (directly stores achievable sums)
- Can be more space-efficient if few sums are achievable
- Easier to find max sum (just use max())

### How It Works
```
Start: dp = {0}
Stone 2: dp = {0, 2}
Stone 3: dp = {0, 2, 3, 5}
Stone 5: dp = {0, 2, 3, 5, 7, 8, 10}
```

## Exercise 2.1: Initialize Set
Create the initial set with base case.

In [ ]:
def initialize_set():
    # TODO: Create a set with only 0 in it
    # This represents that we can achieve sum 0 with empty subset
    pass

# Test
dp_set = initialize_set()
print(f"Initial set: {dp_set}")
print(f"Expected: {0}")

## Exercise 2.2: Add New Sums for One Stone
Given a set of achievable sums and one stone, find all new sums we can create.

In [ ]:
def add_stone_to_set(dp_set, stone, target):
    # TODO: For each sum 's' in dp_set:
    #   If s + stone <= target:
    #       Add (s + stone) to a new_sums set
    # Update dp_set with new_sums
    # Return dp_set
    pass

# Test
dp_set = {0, 2, 3}
target = 10
print(f"Before: {dp_set}")
add_stone_to_set(dp_set, 5, target)
print(f"After adding stone=5: {sorted(dp_set)}")
print(f"Expected: {0, 2, 3, 5, 7, 8} (all combinations)")

## Exercise 2.3: Build Complete Set for All Stones
Process all stones and build the complete set of achievable sums.

In [ ]:
def build_achievable_sums(stones, target):
    # TODO: 
    # 1. Initialize set with {0}
    # 2. For each stone in stones:
    #      Add new sums to the set (only if <= target)
    # 3. Return the set
    pass

# Test
stones = [2, 3, 5]
target = 5
result = build_achievable_sums(stones, target)
print(f"Stones: {stones}, Target: {target}")
print(f"Achievable sums: {sorted(result)}")
print(f"Expected: {0, 2, 3, 5}")

## Exercise 2.4: Complete Set-Based Solution
Implement the full solution using sets.

In [ ]:
def lastStoneWeightII_set(stones):
    # TODO: Implement complete solution
    # 1. Calculate total and target
    # 2. Build set of achievable sums
    # 3. Find max sum in the set
    # 4. Return total - 2 * max_sum
    pass

# Test
print(lastStoneWeightII_set([2,7,4,1,8,1]))  # Expected: 1
print(lastStoneWeightII_set([31,26,33,21,40]))  # Expected: 5

## 📝 Solution 1: DP Array Approach (Reference)

In [ ]:
def lastStoneWeightII_array(stones):
    total = sum(stones)
    target = total // 2
    
    dp = [False] * (target + 1)
    dp[0] = True
    
    for stone in stones:
        for i in range(target, stone - 1, -1):
            dp[i] = dp[i] or dp[i - stone]
    
    for i in range(target, -1, -1):
        if dp[i]:
            return total - 2 * i
    
    return total

## 📝 Solution 2: Set-Based Approach (Reference)

In [ ]:
def lastStoneWeightII_set_solution(stones):
    total = sum(stones)
    target = total // 2
    
    dp = {0}
    
    for stone in stones:
        new_sums = set()
        for s in dp:
            if s + stone <= target:
                new_sums.add(s + stone)
        dp.update(new_sums)
    
    max_sum = max(dp)
    return total - 2 * max_sum

## 🧪 Test Both Solutions

In [ ]:
test_cases = [
    ([2,7,4,1,8,1], 1),
    ([31,26,33,21,40], 5),
    ([1], 1),
    ([5, 5], 0),
    ([1,1,2,3,5,8], 0),
    ([2,2], 0),
    ([1,2], 1)
]

print("Testing Both Approaches:\n")
for stones, expected in test_cases:
    array_result = lastStoneWeightII_array(stones)
    set_result = lastStoneWeightII_set_solution(stones)
    
    match = "✓" if array_result == expected else "✗"
    
    print(f"Input: {stones}")
    print(f"  Array DP: {array_result} {match}")
    print(f"  Set DP:   {set_result} {match}")
    print(f"  Expected: {expected}")
    print()

## 🔍 Detailed Dry Run: DP Array Method

### Input: stones = [2, 4, 1]

**Step 1:** Setup
```
total = 2 + 4 + 1 = 7
target = 7 // 2 = 3
dp = [T, F, F, F]  (indices 0 to 3)
```

**Step 2:** Process stone = 2
```
i=3: dp[3] = dp[3] or dp[1] = F or F = F
i=2: dp[2] = dp[2] or dp[0] = F or T = T ✓

dp = [T, F, T, F]
```

**Step 3:** Process stone = 4 (skip, 4 > target)
```
No updates (stone > target)
dp = [T, F, T, F]
```

**Step 4:** Process stone = 1
```
i=3: dp[3] = dp[3] or dp[2] = F or T = T ✓
i=2: dp[2] = dp[2] or dp[1] = T or F = T
i=1: dp[1] = dp[1] or dp[0] = F or T = T ✓

dp = [T, T, T, T]
```

**Step 5:** Find result
```
max_sum = 3 (dp[3] is True)
result = 7 - 2*3 = 1
```

**Verification:**
- Group 1: [2, 1] = 3
- Group 2: [4] = 4
- Difference: |3 - 4| = 1 ✓

## 🔍 Detailed Dry Run: Set Method

### Input: stones = [2, 4, 1], target = 3

**Step 1:** Initialize
```
dp = {0}
```

**Step 2:** Process stone = 2
```
For s=0: 0+2=2 <= 3 ✓ → add 2
dp = {0, 2}
```

**Step 3:** Process stone = 4
```
For s=0: 0+4=4 > 3 ✗ → skip
For s=2: 2+4=6 > 3 ✗ → skip
dp = {0, 2} (no change)
```

**Step 4:** Process stone = 1
```
For s=0: 0+1=1 <= 3 ✓ → add 1
For s=2: 2+1=3 <= 3 ✓ → add 3
dp = {0, 1, 2, 3}
```

**Step 5:** Calculate result
```
max_sum = max({0, 1, 2, 3}) = 3
result = 7 - 2*3 = 1
```

## Exercise 2.5: Visualize Set Growth
Print how the set grows as we process each stone.

In [ ]:
def visualize_set_growth(stones, target):
    # TODO: Implement and print set after each stone
    dp = {0}
    print(f"Initial: {sorted(dp)}")
    
    # For each stone:
    #   Add new sums
    #   Print current state
    pass

# Test
stones = [2, 4, 1]
target = 3
visualize_set_growth(stones, target)
# Expected output:
# Initial: [0]
# After stone 2: [0, 2]
# After stone 4: [0, 2]
# After stone 1: [0, 1, 2, 3]

## Exercise 2.6: Set vs Array Comparison
Compare memory usage between set and array approaches.

In [ ]:
def compare_space_usage(stones):
    # TODO: 
    # 1. Calculate target
    # 2. Build set-based dp
    # 3. Print: len(set) vs (target + 1)
    # 4. Show which is more space efficient
    pass

# Test
compare_space_usage([2, 7, 4, 1, 8, 1])
# Should show: Set stores only achievable sums, array stores all indices

## 🎯 Advanced Exercise: Track the Partition
Modify the solution to return which stones go into which group.

In [ ]:
def lastStoneWeightII_with_partition(stones):
    # TODO: Instead of storing True/False or just sums,
    # store the actual subsets that achieve each sum
    # dp[sum] = list of stone indices that create this sum
    
    total = sum(stones)
    target = total // 2
    
    # dp[sum] = set of indices used to achieve this sum
    dp = {0: set()}  # sum 0 with empty set
    
    # TODO: Complete the implementation
    pass

# Test
stones = [2,7,4,1,8,1]
result, group1, group2 = lastStoneWeightII_with_partition(stones)
print(f"Stones: {stones}")
print(f"Group 1: {group1} = {sum(group1)}")
print(f"Group 2: {group2} = {sum(group2)}")
print(f"Difference: {result}")

## 📊 Complexity Comparison

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **DP Array** | O(n × target) | O(target) | Fixed space, cache-friendly | May waste space |
| **DP Set** | O(n × target) | O(achievable sums) | Only stores reachable sums | Set operations overhead |

Both are correct and optimal! Choose based on preference.

## 💡 Key Takeaways

1. **Problem Transformation:** Stone smashing → Partition into two groups → Subset sum
2. **DP State:** Track all achievable sums up to target
3. **Backward Iteration:** Prevents using same stone twice
4. **Formula:** Result = Total - 2 × MaxSum
5. **Two Implementations:** Array (boolean) vs Set (actual sums)

## Related Problems
- LeetCode 416: Partition Equal Subset Sum
- LeetCode 494: Target Sum
- LeetCode 698: Partition to K Equal Sum Subsets